# AI Sentinel - Full Fine-Tuning on Colab

This notebook performs full fine-tuning of the violence detection model.
It is separate from the probe notebook to keep things clean.

**WARNING**: Training is disabled by default. Set exactly one of `RUN_STAGE_1_SANITY = True` or `RUN_FULL_TRAINING = True` only after user approval.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Check Dataset and Manifest

In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/content/drive/MyDrive/violence_data')
MANIFEST = Path('/content/drive/MyDrive/AI-Sentinel/manifests/violence_data_manifest.jsonl')

print('Dataset exists:', DATASET_ROOT.exists())
print('Manifest exists:', MANIFEST.exists())

if MANIFEST.exists():
    lines = MANIFEST.read_text().strip().split('\n')
    print('Manifest entries:', len(lines))
    
    import json
    labels = set(json.loads(l)['label'] for l in lines)
    print('Labels:', labels)

## Step 3: Extract Project Code

In [ ]:
import zipfile, shutil
from pathlib import Path

ZIP_CANDIDATES = [
    Path('/content/drive/MyDrive/AI-Sentinel/code/ai-sentinel-colab-code.zip'),
    Path('/content/drive/MyDrive/AI-Sentinel/code/ai-sentinel-code.zip'),
]

ZIP_PATH = None
for candidate in ZIP_CANDIDATES:
    if candidate.exists():
        ZIP_PATH = candidate
        break

print('ZIP_PATH:', ZIP_PATH)
print('ZIP exists:', ZIP_PATH is not None and ZIP_PATH.exists())

if ZIP_PATH is None or not ZIP_PATH.exists():
    raise FileNotFoundError('Project zip not found in Drive')

PROJECT_DIR = Path('/content/ai-sentinel')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(PROJECT_DIR)

print('Project extracted to:', PROJECT_DIR)

## Step 4: Install Dependencies

In [ ]:
import importlib.util

required_packages = {
    'torch': 'torch',
    'torchvision': 'torchvision',
    'cv2': 'opencv-python-headless',
    'numpy': 'numpy',
    'yaml': 'pyyaml',
    'pytorchvideo': 'pytorchvideo',
}

missing_packages = [pip_name for mod, pip_name in required_packages.items() if importlib.util.find_spec(mod) is None]

if missing_packages:
    print(f'Installing: {missing_packages}')
    !pip install -q {' '.join(missing_packages)}
    print('Installation complete.')
else:
    print('All packages already installed.')

# Verify
for mod in required_packages:
    status = 'OK' if importlib.util.find_spec(mod) is not None else 'MISSING'
    print(f'{mod}: {status}')

## Step 5: GPU Check

In [ ]:
import torch
import os

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    print('GPU:', gpu_name)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Total GPU memory: {total_mem:.2f} GB')
    recommended_bs = 4 if total_mem >= 16 else (2 if total_mem >= 8 else 1)
else:
    device = 'cpu'
    gpu_name = 'N/A'
    recommended_bs = 2
    print('WARNING: Using CPU - training will be slow!')

print(f'\nRecommended device: {device}')
print(f'Recommended batch size: {recommended_bs}')

os.environ['CUDA_VISIBLE_DEVICES'] = '0' if device == 'cuda' else ''

## Step 6: Training Configuration

In [ ]:
# Training parameters for post-Stage-1 runs - review before running
TRAINING_CONFIG = {
    'stage_name': 'stage2_controlled',  # Change to 'stage3_full' only after Stage 2 approval
    'epochs': 1,
    'batch_size': 2,
    'learning_rate': 3e-5,
    'max_train_samples': 128,
    'max_val_samples': 128,
    'unfreeze_backbone': False,
}

print('Training Configuration:')
for k, v in TRAINING_CONFIG.items():
    print(f'  {k}: {v}')

print('\nStage 1 sanity runs ignore this block and enforce exactly 32 train / 32 val / 1 epoch.')

## Step 7: Guarded Training Run

In [ ]:
RUN_STAGE_1_SANITY = False  # Set to True only after user approval
RUN_FULL_TRAINING = False  # Set to True only after user approval

if RUN_STAGE_1_SANITY and RUN_FULL_TRAINING:
    raise RuntimeError('Choose either Stage 1 sanity or a later approved run, not both.')
if not RUN_STAGE_1_SANITY and not RUN_FULL_TRAINING:
    raise RuntimeError(
        'Training is disabled. '
        'Set exactly one of RUN_STAGE_1_SANITY=True or RUN_FULL_TRAINING=True only after user approval.'
    )

import os, subprocess, time
from datetime import datetime
from pathlib import Path

os.chdir('/content/ai-sentinel')

env = os.environ.copy()
env['PYTHONPATH'] = 'backend'

CHECKPOINT_ROOT = Path('/content/drive/MyDrive/AI-Sentinel/checkpoints/full_finetune')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
timestamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')

if RUN_STAGE_1_SANITY:
    run_name = f'stage1_sanity_{timestamp}'
else:
    run_name = f"{TRAINING_CONFIG['stage_name']}_{timestamp}"

RUN_OUTPUT_DIR = CHECKPOINT_ROOT / run_name

cmd = [
    'python', 'backend/train_finetune.py',
    '--manifest', str(MANIFEST),
    '--weights', 'backend/best_model.pt',
    '--output-dir', str(RUN_OUTPUT_DIR),
    '--cache-dir', '/content/ai-sentinel/.runlogs/training/cache',
    '--batch-size', str(TRAINING_CONFIG['batch_size']),
    '--seed', '42',
    '--num-workers', '2',
]

if RUN_STAGE_1_SANITY:
    cmd.extend([
        '--epochs', '1',
        '--lr', '1e-4',
        '--max-train-samples', '32',
        '--max-val-samples', '32',
    ])
else:
    cmd.extend([
        '--epochs', str(TRAINING_CONFIG['epochs']),
        '--lr', str(TRAINING_CONFIG['learning_rate']),
    ])
    if TRAINING_CONFIG['max_train_samples'] > 0:
        cmd.extend(['--max-train-samples', str(TRAINING_CONFIG['max_train_samples'])])
    if TRAINING_CONFIG['max_val_samples'] > 0:
        cmd.extend(['--max-val-samples', str(TRAINING_CONFIG['max_val_samples'])])
    if TRAINING_CONFIG['unfreeze_backbone']:
        cmd.append('--unfreeze-backbone')

print(f"Running: {' '.join(cmd)}")
start = time.time()
result = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=7200)
runtime = time.time() - start

print(f'\nTraining completed in {runtime:.2f}s')
print(f'Return code: {result.returncode}')
print(f'Run output dir: {RUN_OUTPUT_DIR}')
print(f'\nSTDOUT (last 4000 chars):\n{result.stdout[-4000:]}')
if result.stderr:
    print(f'\nSTDERR (last 4000 chars):\n{result.stderr[-4000:]}')

if result.returncode != 0:
    raise RuntimeError('Training failed. Check stderr above.')

## Step 8: Parse Results

In [ ]:
from pathlib import Path
import json

CHECKPOINT_ROOT = Path('/content/drive/MyDrive/AI-Sentinel/checkpoints/full_finetune')
run_output_dir = globals().get('RUN_OUTPUT_DIR')

candidate_paths = []
if run_output_dir is not None:
    candidate_paths.append(Path(run_output_dir) / 'best_candidate.pt')
candidate_paths.extend(sorted(
    CHECKPOINT_ROOT.rglob('best_candidate.pt'),
    key=lambda p: p.stat().st_mtime,
    reverse=True
))
all_checkpoints = [path for path in candidate_paths if path.exists()]
latest = all_checkpoints[0] if all_checkpoints else None

if latest is not None:
    print('Latest checkpoint:', latest)
    
    import torch
    checkpoint = torch.load(latest, map_location='cpu')
    
    print('\nCheckpoint metrics:')
    if 'metrics' in checkpoint:
        print(json.dumps(checkpoint['metrics'], indent=2))
    
    print('\nTraining args:')
    if 'trainArgs' in checkpoint:
        print(json.dumps(checkpoint['trainArgs'], indent=2))
    
    print('\nHistory (last 3 epochs):')
    if 'history' in checkpoint:
        for epoch_data in checkpoint['history'][-3:]:
            print(json.dumps(epoch_data))
else:
    print('No checkpoints found.')

## Step 9: Save Report

In [ ]:
from pathlib import Path
import json
from datetime import datetime

report_dir = Path('/content/drive/MyDrive/AI-Sentinel/reports/')
report_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir = globals().get('RUN_OUTPUT_DIR', Path('/content/drive/MyDrive/AI-Sentinel/checkpoints/full_finetune'))
latest_checkpoint = latest if 'latest' in globals() and latest is not None else 'N/A'

report = f"""# Full Fine-Tuning Report

## Execution Status
- Notebook executed: Yes
- Device: {device}
- GPU: {gpu_name}
- Training config: {TRAINING_CONFIG}

## Dataset
- Path: `{DATASET_ROOT}`
- Manifest: `{MANIFEST}`

## Results
- Checkpoint dir: `{checkpoint_dir}`
- Latest checkpoint: `{latest_checkpoint}`

## Safety
- Production weights overwritten: No
- Checkpoints saved to Drive: Yes

---
*Report generated: {datetime.now().isoformat()}*
"""

report_path = report_dir / 'full_finetune_report.md'
report_path.write_text(report)

print('Report saved to:', report_path)
print('\nREPORT_CONTENT:')
print('='*60)
print(report)
print('='*60)